Install Python

In [ ]:
!pip install -q pymupdf sentence-transformers transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 26.4 MB/s eta 0:00:00


Import libraries

In [ ]:
import pymupdf
import re
import numpy as np
import pandas as pd
import torch

from sentence_transformers import SentenceTransformer
from transformers import pipeline

Upload the PDF

In [ ]:
from google.colab import files

# Upload the PDF file
uploaded = files.upload()

# Get the path of the uploaded file
if uploaded:
    file_path = next(iter(uploaded))
    print(f"File '{file_path}' uploaded successfully.")
else:
    file_path = None
    print("No file uploaded.")

Saving Business.pdf to Business.pdf
File 'Business.pdf' uploaded successfully.


In [ ]:
pdf_path = list(uploaded.keys())[0]

print("", pdf_path)

 Business.pdf


Read the PDF

In [ ]:
def read_pdf(pdf_path):

    document = pymupdf.open(pdf_path)

    pages_and_texts = []

    for page_number, page in enumerate(document):

        text = page.get_text("text")

        pages_and_texts.append({
            "page_number": page_number + 1,
            "text": text
        })

    return pages_and_texts


pages_and_texts = read_pdf(pdf_path)

print("Total PDF pages:", len(pages_and_texts))

Total PDF pages: 1001


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Check PDF extraction

In [ ]:
print(pages_and_texts[8]["text"][:1500])

viii 
 Contents
 
 13.4 Making Inferences About Population Parameters 684
 
 13.5 Modeling Techniques 703
 
  
Statistics at Work 719
 
  
Chapter Review 720
 
  
Flow Chart: Multiple Regression and Modeling 731
CHAPTER 14 
Nonparametric Methods 733
 
 14.1 Introduction to Nonparametric Statistics 734
 
 14.2 The Sign Test for Paired Data 736
 
 14.3  Rank Sum Tests: The Mann–Whitney U Test  
DQGWKH.UXVNDO±:DOOLV7HVW 
 
 14.4 The One-sample Runs Test 758
 
 14.5 Rank Correlation 767

  7KH.ROPRJRURY±6PLUQRY7HVW 
 
  
Statistics at Work 786
 
  
Chapter Review 787
 
  
Flow Chart: Nonparametric Methods 800
CHAPTER 15 
Time Series and Forecasting 803
 
 15.1 Introduction 804
 
 15.2 Variations in Time Series 804
 
 15.3 Trend Analysis 806
 
 15.4 Cyclical Variation 818
 
 15.5 Seasonal Variation 824
 
 15.6 Irregular Variation 833
 
 15.7  A Problem Involving All Four Components of a Time Series 834
 
 15.8 Time-Series Analysis in Forecasting 844
 
  
Statistics at Wo

Clean the extracted text

In [ ]:
def clean_text(text):

    text = text.replace("\n", " ")

    text = re.sub(r"\s+", " ", text)

    text = text.replace("￾", "")

    return text.strip()


for item in pages_and_texts:

    item["text"] = clean_text(item["text"])

In [ ]:
print(pages_and_texts[8]["text"][:1000])

viii Contents 13.4 Making Inferences About Population Parameters 684 13.5 Modeling Techniques 703 Statistics at Work 719 Chapter Review 720 Flow Chart: Multiple Regression and Modeling 731 CHAPTER 14 Nonparametric Methods 733 14.1 Introduction to Nonparametric Statistics 734 14.2 The Sign Test for Paired Data 736 14.3 Rank Sum Tests: The Mann–Whitney U Test DQGWKH.UXVNDO±:DOOLV7HVW  14.4 The One-sample Runs Test 758 14.5 Rank Correlation 767    7KH.ROPRJRURY±6PLUQRY7HVW  Statistics at Work 786 Chapter Review 787 Flow Chart: Nonparametric Methods 800 CHAPTER 15 Time Series and Forecasting 803 15.1 Introduction 804 15.2 Variations in Time Series 804 15.3 Trend Analysis 806 15.4 Cyclical Variation 818 15.5 Seasonal Variation 824 15.6 Irregular Variation 833 15.7 A Problem Involving All Four Components of a Time Series 834 15.8 Time-Series Analysis in Forecasting 844 Statistics at Work 844 Chapter Review 846 Flow Chart: Time Series 853


Split text into sentences

In [ ]:
def split_into_sentences(text):

    sentences = re.split(
        r'(?<=[.!?])\s+',
        text
    )

    sentences = [
        sentence.strip()
        for sentence in sentences
        if sentence.strip()
    ]

    return sentences

In [ ]:
for item in pages_and_texts:

    item["sentences"] = split_into_sentences(
        item["text"]
    )

    item["sentence_count"] = len(
        item["sentences"]
    )

In [ ]:
pages_and_texts[8]["sentences"][:5]

['viii Contents 13.4 Making Inferences About Population Parameters 684 13.5 Modeling Techniques 703 Statistics at Work 719 Chapter Review 720 Flow Chart: Multiple Regression and Modeling 731 CHAPTER 14 Nonparametric Methods 733 14.1 Introduction to Nonparametric Statistics 734 14.2 The Sign Test for Paired Data 736 14.3 Rank Sum Tests: The Mann–Whitney U Test DQG\x03WKH\x03.UXVNDO±:DOOLV\x037HVW\x03 \x1a\x17\x17 14.4 The One-sample Runs Test 758 14.5 Rank Correlation 767 \x03 \x03 \x14\x17\x11\x19\x03 7KH\x03.ROPRJRURY±6PLUQRY\x037HVW\x03 \x1a\x1a Statistics at Work 786 Chapter Review 787 Flow Chart: Nonparametric Methods 800 CHAPTER 15 Time Series and Forecasting 803 15.1 Introduction 804 15.2 Variations in Time Series 804 15.3 Trend Analysis 806 15.4 Cyclical Variation 818 15.5 Seasonal Variation 824 15.6 Irregular Variation 833 15.7 A Problem Involving All Four Components of a Time Series 834 15.8 Time-Series Analysis in Forecasting 844 Statistics at Work 844 Chapter Review 846 Flow

Create sentence chunks

In [ ]:
NUM_SENTENCES_PER_CHUNK = 8


def create_sentence_chunks(
    sentences,
    chunk_size=NUM_SENTENCES_PER_CHUNK
):

    chunks = []

    for i in range(
        0,
        len(sentences),
        chunk_size
    ):

        chunk = sentences[
            i:i + chunk_size
        ]

        chunks.append(chunk)

    return chunks

In [ ]:
for item in pages_and_texts:

    item["sentence_chunks"] = (
        create_sentence_chunks(
            item["sentences"]
        )
    )

Flatten everything into individual chunks

In [ ]:
pages_and_chunks = []


for item in pages_and_texts:

    for chunk in item["sentence_chunks"]:

        chunk_text = " ".join(chunk)

        pages_and_chunks.append({
            "page_number":
                item["page_number"],

            "sentence_chunk":
                chunk_text,

            "chunk_word_count":
                len(chunk_text.split()),

            "chunk_char_count":
                len(chunk_text)
        })

In [ ]:
print(
    "Total chunks:",
    len(pages_and_chunks)
)

Total chunks: 1937


Convert chunks to DataFrame

In [ ]:
df = pd.DataFrame(
    pages_and_chunks
)

df.head()

,page_number,sentence_chunk,chunk_word_count,chunk_char_count
0,2,MANAGEMENT S t a t i s t i c s f o r Richard I...,55,300
1,3,ISBN 978-93-325-8118-0 Copyright © 2017 Pearso...,103,649
2,3,"Registered Office: 4th Floor, Software Block, ...",32,253
3,4,Contents Preface xi CHAPTER 1 Introduction 1 1...,137,833
4,5,iv Contents 3.3 A Second Measure of Central Te...,198,1319


check

In [ ]:
print(df.shape)

df.head(10)

(1937, 4)


,page_number,sentence_chunk,chunk_word_count,chunk_char_count
0,2,MANAGEMENT S t a t i s t i c s f o r Richard I...,55,300
1,3,ISBN 978-93-325-8118-0 Copyright © 2017 Pearso...,103,649
2,3,"Registered Office: 4th Floor, Software Block, ...",32,253
3,4,Contents Preface xi CHAPTER 1 Introduction 1 1...,137,833
4,5,iv Contents 3.3 A Second Measure of Central Te...,198,1319
5,6,Contents v 5.6 The Normal Distribution: A Dist...,171,1171
6,7,vi Contents CHAPTER 8 Testing Hypotheses: One-...,198,1338
7,8,Contents vii 10.6 Total Quality Management 494...,173,1169
8,9,viii Contents 13.4 Making Inferences About Pop...,145,964
9,10,Contents ix CHAPTER 16 Index Numbers 855   ...,112,780


Estimate token count

In [ ]:
df["chunk_token_count"] = (
    df["chunk_char_count"] / 4
)

Check

In [ ]:
df[
    [
        "page_number",
        "chunk_word_count",
        "chunk_token_count"
    ]
].describe()

,page_number,chunk_word_count,chunk_token_count
count,1937.000000,1937.000000,1937.000000
mean,465.667011,161.486835,269.906557
std,277.020191,92.459110,160.215938
min,2.000000,1.000000,2.000000
25%,226.000000,104.000000,167.000000
50%,454.000000,157.000000,257.000000
75%,696.000000,209.000000,347.000000
max,1000.000000,1053.000000,1411.500000


Remove extremely small chunks

In [ ]:
MIN_TOKEN_LENGTH = 20

df_filtered = df[df["chunk_token_count"] >= MIN_TOKEN_LENGTH].reset_index(drop=True)
df_filtered.sample(5)

,page_number,sentence_chunk,chunk_word_count,chunk_char_count,chunk_token_count
1577,790,Using the correlation techniques of Chapter 12...,87,679,169.75
951,457,Both samples are large enough to justify using...,167,974,243.50
1851,967,p n r 0.01 0.02 0.03 0.04 0.05 0.06 0.07 0.08 ...,653,3374,843.50
745,352,To change this number of successes to the prop...,164,946,236.50
622,298,Sampling and Sampling Distributions 283 What I...,303,1995,498.75


In [ ]:
MIN_TOKEN_LENGTH = 20

df = df[
    df["chunk_token_count"]
    >= MIN_TOKEN_LENGTH
].reset_index(drop=True)

print(
    "Chunks remaining:",
    len(df)
)

Chunks remaining: 1896


Safe random sampling

In [ ]:
sample_size = min(
    5,
    len(df)
)

df.sample(
    n=sample_size,
    random_state=42
)[
    [
        "page_number",
        "chunk_token_count",
        "sentence_chunk"
    ]
]

,page_number,chunk_token_count,sentence_chunk
1319,640,427.00,Simple Regression and Correlation 625 G  +RX...
611,292,147.00,Sampling and Sampling Distributions 277 Drawin...
964,465,473.00,450 Statistics for Management The lower limit ...
1316,636,271.50,This justifies our use of se to compute approx...
585,280,64.25,231 λx × e–λ You may use the Poisson distribut...


Load embedding model

In [ ]:
"all-MiniLM-L6-v2"

'all-MiniLM-L6-v2'

In [ ]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Test one embedding

In [ ]:
sample_text = (
    "Retrieval augmented generation combines "
    "retrieval with text generation."
)

sample_embedding = (
    embedding_model.encode(sample_text)
)

print(
    sample_embedding.shape
)

(384,)


Generate embeddings for the entire book

In [ ]:
texts = df[
    "sentence_chunk"
].tolist()


embeddings = embedding_model.encode(

    texts,

    batch_size=32,

    show_progress_bar=True,

    convert_to_numpy=True,

    normalize_embeddings=True
)

Batches:   0%|          | 0/60 [00:00<?, ?it/s]

check

In [ ]:
print(
    embeddings.shape
)

(1896, 384)


Store embeddings

In [ ]:
df["embedding"] = list(
    embeddings
)

In [ ]:
df.head()

,page_number,sentence_chunk,chunk_word_count,chunk_char_count,chunk_token_count,embedding
0,2,MANAGEMENT S t a t i s t i c s f o r Richard I...,55,300,75.00,"[-0.022275323, 0.016915604, -0.08190675, 0.023..."
1,3,ISBN 978-93-325-8118-0 Copyright © 2017 Pearso...,103,649,162.25,"[0.056866296, 0.03099257, 0.0071852007, 0.0423..."
2,3,"Registered Office: 4th Floor, Software Block, ...",32,253,63.25,"[-0.105907016, -0.048195876, -0.033170156, -0...."
3,4,Contents Preface xi CHAPTER 1 Introduction 1 1...,137,833,208.25,"[0.10056515, -0.001954792, -0.020985132, 0.006..."
4,5,iv Contents 3.3 A Second Measure of Central Te...,198,1319,329.75,"[-0.006064098, -0.06444124, 0.03597292, -0.033..."


Save processed RAG database

In [ ]:
df.to_pickle(
    "genai_book_embeddings.pkl"
)

In [ ]:
from google.colab import files

files.download(
    "genai_book_embeddings.pkl"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Create query embedding

In [ ]:
query = (
    "What is retrieval augmented generation?"
)

query_embedding = (
    embedding_model.encode(
        query,
        normalize_embeddings=True
    )
)

Semantic similarity search

In [ ]:
scores = np.dot(
    embeddings,
    query_embedding
)

In [ ]:
top_k = 5

top_indices = np.argsort(
    scores
)[-top_k:][::-1]

Display retrieved chunks

In [ ]:
for rank, index in enumerate(
    top_indices,
    start=1
):

    print(
        "\nRESULT",
        rank
    )

    print(
        "Page:",
        df.iloc[index][
            "page_number"
        ]
    )

    print(
        "Similarity:",
        round(
            float(scores[index]),
            3
        )
    )

    print(
        df.iloc[index][
            "sentence_chunk"
        ]
    )

    print(
        "=" * 80
    )


RESULT 1
Page: 294
Similarity: 0.284
Judgement Sampling Researcher excesises his/her judgement to draw a sample which he/she thinks is representative of the population or otherwise appropriate. It is also known as Purposive Sampling. It is better than the previous one but personal bias limits the applicability of this sampling scheme. Quota Sampling ,WFRQVLVWVRI¿[DWLRQRIFHUWDLQTXRWDVRQWKHEDVLVRIFHUWDLQSDUDPHWHU V VR as to make sample representative of the population under study. It is one of the most commonly used QRQUDQGRPVDPSOLQJ,WLVPXFKVLPLODUWRVWUDWL¿HGUDQGRPVDPSOLQJEXWGRHVQRWUHTXLUHFRPSOHWH list of population elements. Shopping-Mall Intercept Sampling This sample involves drawing samples (establishing malls) in market-places, shopping malls, fairs in different socioeconomic locations, so to make

RESULT 2
Page: 466
Similarity: 0.262
In other words, the prob value is the probability Deciding before we take a sample Prob values Another advantage Sett

In [ ]:
What is retrieval augmented generation?

Object `generation` not found.


Turn retrieval into a reusable function

In [ ]:
def retrieve_relevant_chunks(
    query,
    top_k=5
):

    query_embedding = (
        embedding_model.encode(
            query,
            normalize_embeddings=True
        )
    )

    scores = np.dot(
        embeddings,
        query_embedding
    )

    top_indices = np.argsort(
        scores
    )[-top_k:][::-1]

    results = []

    for index in top_indices:

        results.append({

            "page_number":
                int(
                    df.iloc[index][
                        "page_number"
                    ]
                ),

            "score":
                float(
                    scores[index]
                ),

            "text":
                df.iloc[index][
                    "sentence_chunk"
                ]

        })

    return results

Load a small generation model

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base", device_map="auto")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Check GPU

In [ ]:
print(
    "GPU available:",
    torch.cuda.is_available()
)

GPU available: False


Build the RAG prompt

In [ ]:
def build_prompt(
    query,
    retrieved_chunks
):

    context = ""

    for item in retrieved_chunks:

        context += (
            f"\nPage "
            f"{item['page_number']}:\n"
            f"{item['text']}\n"
        )

    prompt = f"""
You are an MBA teaching assistant.

Answer the question using only the
context provided below.

If the answer is not available in the
context, say:

"I could not find this information
in the book."

Explain the answer in simple language.

Context:

{context}

Question:

{query}

Answer:
"""

    return prompt

Complete RAG generation

In [ ]:
def ask_book(
    query,
    top_k=5
):

    retrieved_chunks = (
        retrieve_relevant_chunks(
            query,
            top_k=top_k
        )
    )

    prompt = build_prompt(
        query,
        retrieved_chunks
    )

    # Encode the prompt for the T5 model, truncating if too long
    inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True).to(model.device)

    # Generate output using the model directly
    outputs = model.generate(
        **inputs,
        max_new_tokens=500,
        do_sample=False
    )

    # Decode the generated tokens
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract only the generated answer part, removing the prompt
    answer_start = generated_text.find("Answer:")
    if answer_start != -1:
        # Add len("Answer:") + 1 to get past the 'Answer:' itself and the newline
        answer = generated_text[answer_start + len("Answer:") + 1:].strip()
    else:
        answer = generated_text.strip() # Fallback if 'Answer:' isn't found

    return (
        answer,
        retrieved_chunks
    )

In [ ]:
print('--- Re-testing RAG with Sample Questions ---')

questions = [

    "What is generative AI?",

    "What is the difference between traditional AI and generative AI?",

    "What is a large language model?",

    "What are vector embeddings?",

    "What is retrieval augmented generation?",

    "Why is data important for generative AI?",

    "What is prompt engineering?",

    "What is in-context learning?",

    "What is fine tuning?",

    "What factors should a company consider while selecting an LLM?"

]

# Iterate through a few questions to test the RAG
# You can modify the slice [:] to test more or fewer questions
for i, question in enumerate(questions):
    print(f"\nQUESTION {i+1}: {question}")

    answer, sources = ask_book(question)

    print(f"ANSWER: {answer}")

    print("\nSOURCES (Top 5 Retrieved Chunks):")
    if sources:
        for j, source in enumerate(sources):
            print(f"  Source {j+1}: Page {source['page_number']}, Similarity: {round(source['score'], 3)}")
            print(f"    Text Snippet: {source['text'][:200]}...") # Print first 200 chars
    else:
        print("  No sources retrieved.")

    print("=" * 100)

--- Re-testing RAG with Sample Questions ---

QUESTION 1: What is generative AI?
ANSWER: Robots are more efficient at detecting and detecting changes in the environment.

SOURCES (Top 5 Retrieved Chunks):
  Source 1: Page 213, Similarity: 0.259
    Text Snippet: Probability Tree A graphical representation showing the possible outcomes of a series of experiments and their respective probabilities. Relative Frequency of Occurrence The proportion of times that a...
  Source 2: Page 484, Similarity: 0.256
    Text Snippet: Quality and Quality Control 469 War II, Japanese managers incorporated Deming’s ideas into their management philosophy. Many American industries, including automobiles and consumer electronics, encoun...
  Source 3: Page 485, Similarity: 0.255
    Text Snippet: 470 Statistics for Management EXERCISES 10.2 Basic Concepts 10-7 What happened in the 1970s and 1980s to cause American managers to pay more attention to Deming’s ideas? 10-8 Explain why work produced...
  Source 

In [ ]:
sample_question = "What is generative AI?"
print(f"QUESTION: {sample_question}")

answer, sources = ask_book(sample_question)

print(f"ANSWER: {answer}")
print("\nSOURCES:")
for source in sources:
    print(f"  Page: {source['page_number']}, Similarity: {round(source['score'], 3)}, Text: {source['text'][:100]}...")

QUESTION: What is generative AI?
ANSWER: Robots are more efficient at detecting and detecting changes in the environment.

SOURCES:
  Page: 213, Similarity: 0.259, Text: Probability Tree A graphical representation showing the possible outcomes of a series of experiments...
  Page: 484, Similarity: 0.256, Text: Quality and Quality Control 469 War II, Japanese managers incorporated Deming’s ideas into their man...
  Page: 485, Similarity: 0.255, Text: 470 Statistics for Management EXERCISES 10.2 Basic Concepts 10-7 What happened in the 1970s and 1980...
  Page: 589, Similarity: 0.212, Text: (c) State an explicit conclusion. 11-41 MacroSwift has designed a new operating system that will rev...
  Page: 466, Similarity: 0.211, Text: In other words, the prob value is the probability Deciding before we take a sample Prob values Anoth...


Ask the book a question

In [ ]:
answer, sources = ask_book(
    "What are vector embeddings?"
)

print(answer)

trend component from the time series is a good way to study the other three components of the time series.


Show sources

In [ ]:
for source in sources:

    print(
        "\nPage:",
        source["page_number"]
    )

    print(
        "Similarity:",
        round(
            source["score"],
            3
        )
    )

    print(
        source["text"]
    )


Page: 213
Similarity: 0.274
Probability Tree A graphical representation showing the possible outcomes of a series of experiments and their respective probabilities. Relative Frequency of Occurrence The proportion of times that an event occurs in the long run when conditions are stable, or the observed relative frequency of an event in a very large number of trials. Sample Space The set of all possible outcomes of an experiment. Statistical Dependence The condition when the probability of some event is dependent on, or affected by, the occurrence of some other event. Statistical Independence The condition when the occurrence of one event has no effect on the prob- ability of occurrence of another event. 6XEMHFWLYH3UREDELOLW\ Probabilities based on the personal beliefs of the person making the prob- ability estimate. Venn Diagram A pictorial representation of probability concepts in which the sample space is repre- sented as a rectangle and the events in the sample space as portions of

Try MBA-friendly questions

In [ ]:
questions = [

    "What is generative AI?",

    "What is the difference between traditional AI and generative AI?",

    "What is a large language model?",

    "What are vector embeddings?",

    "What is retrieval augmented generation?",

    "Why is data important for generative AI?",

    "What is prompt engineering?",

    "What is in-context learning?",

    "What is fine tuning?",

    "What factors should a company consider while selecting an LLM?"

]

In [ ]:
for question in questions[:3]:

    print(
        "\nQUESTION:",
        question
    )

    answer, _ = ask_book(
        question
    )

    print(
        "ANSWER:",
        answer
    )

    print(
        "=" * 80
    )


QUESTION: What is generative AI?
ANSWER: Robots are more efficient at detecting and detecting changes in the environment.

QUESTION: What is the difference between traditional AI and generative AI?
ANSWER: American industries.

QUESTION: What is a large language model?
ANSWER: QRW3UHDOPDQXIDFWXULQJ


BOOK PDF
   ↓
INGESTION
   ↓
TEXT EXTRACTION
   ↓
TEXT CLEANING
   ↓
SENTENCE SPLITTING
   ↓
CHUNKING
   ↓
EMBEDDING MODEL
   ↓
VECTOR EMBEDDINGS
   ↓
USER QUESTION
   ↓
QUESTION EMBEDDING
   ↓
SIMILARITY SEARCH
   ↓
TOP RELEVANT CHUNKS
   ↓
PROMPT + CONTEXT
   ↓
LLM
   ↓
ANSWER

In [ ]:
question = "What is large language model?" # Define question explicitly for this cell
print(
    "QUESTION:",
    question
)

answer, _ = ask_book(
    question
)

print(
    "ANSWER:",
    answer
)

QUESTION: What is large language model?
ANSWER: QRWWRPHQWLRQZKDWLWPXVWEHGRLQJWRRXU


In [ ]:

!pip install -q pymupdf sentence-transformers transformers accelerate

### Testing the RAG System

To test the RAG system, you can ask a series of questions and then evaluate the generated answers along with the retrieved sources. This allows you to check for:

1.  **Relevance of Retrieved Chunks**: Are the `sources` (page number, similarity score, and text snippet) directly related to the question?
2.  **Accuracy of Answer**: Does the `answer` correctly synthesize information from the provided `sources`?
3.  **Completeness**: Does the answer cover the main points of the question based on the available context?
4.  **Coherence**: Is the answer well-written and easy to understand?

In [ ]:
print('--- Testing RAG with Sample Questions ---')

# Iterate through a few questions to test the RAG
# You can modify the slice [:] to test more or fewer questions
for i, question in enumerate(questions):
    print(f"\nQUESTION {i+1}: {question}")

    answer, sources = ask_book(question)

    print(f"ANSWER: {answer}")

    print("\nSOURCES (Top 5 Retrieved Chunks):")
    if sources:
        for j, source in enumerate(sources):
            print(f"  Source {j+1}: Page {source['page_number']}, Similarity: {round(source['score'], 3)}")
            print(f"    Text Snippet: {source['text'][:200]}...") # Print first 200 chars
    else:
        print("  No sources retrieved.")

    print("=" * 100)

--- Testing RAG with Sample Questions ---

QUESTION 1: What is generative AI?
ANSWER: Robots are more efficient at detecting and detecting changes in the environment.

SOURCES (Top 5 Retrieved Chunks):
  Source 1: Page 213, Similarity: 0.259
    Text Snippet: Probability Tree A graphical representation showing the possible outcomes of a series of experiments and their respective probabilities. Relative Frequency of Occurrence The proportion of times that a...
  Source 2: Page 484, Similarity: 0.256
    Text Snippet: Quality and Quality Control 469 War II, Japanese managers incorporated Deming’s ideas into their management philosophy. Many American industries, including automobiles and consumer electronics, encoun...
  Source 3: Page 485, Similarity: 0.255
    Text Snippet: 470 Statistics for Management EXERCISES 10.2 Basic Concepts 10-7 What happened in the 1970s and 1980s to cause American managers to pay more attention to Deming’s ideas? 10-8 Explain why work produced...
  Source 4: 

In [ ]:
question = "What is mode?"
print("QUESTION:", question)

answer, sources = ask_book(question)

print("ANSWER:", answer)
print("\nSOURCES (Top 5 Retrieved Chunks):")
if sources:
    for j, source in enumerate(sources):
        print(f"  Source {j+1}: Page {source['page_number']}, Similarity: {round(source['score'], 3)}")
        print(f"    Text Snippet: {source['text'][:200]}...") # Print first 200 chars
else:
    print("  No sources retrieved.")

print("=" * 100)

QUESTION: What is mode?
ANSWER: The mode is not unduly affected by extreme values.

SOURCES (Top 5 Retrieved Chunks):
  Source 1: Page 121, Similarity: 0.563
    Text Snippet: Advantages and Disadvantages of the Mode The mode, like the median, can be used as a central location for qualitative as well as quantitative data. If a printing press WXUQVRXW¿YHLPSUHVVLRQVZKLFK...
  Source 2: Page 122, Similarity: 0.465
    Text Snippet: Mode Mean Median Mode Mean Median (a) (b) FIGURE 3-8 POSITIVELY (A) AND NEGATIVELY (B) SKEWED DISTRIBUTIONS, ILLUSTRATING RELATIVE POSITIONS OF MEAN, MEDIAN, AND MODE. Mean, median, and mode are ident...
  Source 3: Page 120, Similarity: 0.446
    Text Snippet: Measures of Central Tendency and Dispersion in Frequency Distributions 105 TABLE 3-13 DELIVERY TRIPS PER DAY IN ONE 20-DAY PERIOD Trips Arrayed in Ascending Order 0 2 5 7 15 0 2 5 7 15 ĸ0RGH 1 4 6 8 ...
  Source 4: Page 122, Similarity: 0.414
    Text Snippet: Measures of Central Tendency and Dispe

In [ ]:
# Necessary imports
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Initialize models (embedding_model is also initialized in c803d73f for robustness)
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base", device_map="auto")

# df and embeddings are expected to be available from previous successful cell executions (e.g., c803d73f).
# Removed explicit loading to avoid FileNotFoundError when the variables are already in scope.

def retrieve_relevant_chunks(
    query,
    top_k=5
):

    query_embedding = (
        embedding_model.encode(
            query,
            normalize_embeddings=True
        )
    )

    scores = np.dot(
        embeddings,
        query_embedding
    )

    top_indices = np.argsort(
        scores
    )[-top_k:][::-1]

    results = []

    for index in top_indices:

        results.append({

            "page_number":
                int(
                    df.iloc[index][
                        "page_number"
                    ]
                ),

            "score":
                float(
                    scores[index]
                ),

            "text":
                df.iloc[index][
                    "sentence_chunk"
                ]

        })

    return results

def build_prompt(
    query,
    retrieved_chunks
):

    context = ""

    for item in retrieved_chunks:

        context += (
            f"\nPage "
            f"{item['page_number']}:\n"
            f"{item['text']}\n"
        )

    prompt = f"""
You are an MBA teaching assistant.

Answer the question using only the
context provided below.

If the answer is not available in the
context, say:

"I could not find this information
in the book."

Explain the answer in simple language.

Context:

{context}

Question:

{query}

Answer:
"""

    return prompt

def ask_book(
    query,
    top_k=5
):

    retrieved_chunks = (
        retrieve_relevant_chunks(
            query,
            top_k=top_k
        )
    )

    prompt = build_prompt(
        query,
        retrieved_chunks
    )

    # Encode the prompt for the T5 model, truncating if too long
    inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True).to(model.device)

    # Generate output using the model directly
    outputs = model.generate(
        **inputs,
        max_new_tokens=500,
        do_sample=False
    )

    # Decode the generated tokens
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract only the generated answer part, removing the prompt
    answer_start = generated_text.find("Answer:")
    if answer_start != -1:
        # Add len("Answer:") + 1 to get past the 'Answer:' itself and the newline
        answer = generated_text[answer_start + len("Answer:") + 1:].strip()
    else:
        answer = generated_text.strip() # Fallback if 'Answer:' isn't found

    return (
        answer,
        retrieved_chunks
    )

# The ask_book function will be called in a subsequent step.

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [ ]:
question = "What is the formula for standard deviation?"
print("QUESTION:", question)

answer, sources = ask_book(question)

print("ANSWER:", answer)
print("\nSOURCES (Top 5 Retrieved Chunks):")
if sources:
    for j, source in enumerate(sources):
        print(f"  Source {j+1}: Page {source['page_number']}, Similarity: {round(source['score'], 3)}")
        print(f"    Text Snippet: {source['text'][:200]}...") # Print first 200 chars
else:
    print("  No sources retrieved.")

print("=" * 100)

QUESTION: What is the formula for standard deviation?


KeyboardInterrupt: 

In [ ]:
# Re-executing d2071f77 to define the RAG functions correctly after c803d73f has populated df and embeddings.

# Necessary imports
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Initialize models (embedding_model is also initialized in c803d73f for robustness)
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base", device_map="auto")

# df and embeddings are expected to be available from previous successful cell executions (e.g., c803d73f).
# Removed explicit loading to avoid FileNotFoundError when the variables are already in scope.

def retrieve_relevant_chunks(
    query,
    top_k=5
):

    query_embedding = (
        embedding_model.encode(
            query,
            normalize_embeddings=True
        )
    )

    scores = np.dot(
        embeddings,
        query_embedding
    )

    top_indices = np.argsort(
        scores
    )[-top_k:][::-1]

    results = []

    for index in top_indices:

        results.append({

            "page_number":
                int(
                    df.iloc[index][
                        "page_number"
                    ]
                ),

            "score":
                float(
                    scores[index]
                ),

            "text":
                df.iloc[index][
                    "sentence_chunk"
                ]

        })

    return results

def build_prompt(
    query,
    retrieved_chunks
):

    context = ""

    for item in retrieved_chunks:

        context += (
            f"\nPage {item['page_number']}:\n"
            f"{item['text']}\n"
        )

    prompt = f"""
You are an MBA teaching assistant.

Answer the question using only the
context provided below.

If the answer is not available in the
context, say:

"I could not find this information
in the book."

Explain the answer in simple language.

Context:

{context}

Question:

{query}

Answer:
"""

    return prompt

def ask_book(
    query,
    top_k=5
):

    retrieved_chunks = (
        retrieve_relevant_chunks(
            query,
            top_k=top_k
        )
    )

    prompt = build_prompt(
        query,
        retrieved_chunks
    )

    # Encode the prompt for the T5 model, truncating if too long
    inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True).to(model.device)

    # Generate output using the model directly
    outputs = model.generate(
        **inputs,
        max_new_tokens=500,
        do_sample=False
    )

    # Decode the generated tokens
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract only the generated answer part, removing the prompt
    answer_start = generated_text.find("Answer:")
    if answer_start != -1:
        # Add len("Answer:") + 1 to get past the 'Answer:' itself and the newline
        answer = generated_text[answer_start + len("Answer:") + 1:].strip()
    else:
        answer = generated_text.strip() # Fallback if 'Answer:' isn't found

    return (
        answer,
        retrieved_chunks
    )

# The ask_book function will be called in a subsequent step.

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [ ]:
# Now asking the original question 'what is mean?' after ensuring all RAG components are defined and data is loaded.
question = "What is mean?"
print("QUESTION:", question)

answer, sources = ask_book(question)

print("ANSWER:", answer)
print("\nSOURCES (Top 5 Retrieved Chunks):")
if sources:
    for j, source in enumerate(sources):
        print(f"  Source {j+1}: Page {source['page_number']}, Similarity: {round(source['score'], 3)}")
        print(f"    Text Snippet: {source['text'][:200]}...") # Print first 200 chars
else:
    print("  No sources retrieved.")

print("=" * 100)

In [ ]:
# Install necessary libraries if not already installed within the environment
!pip install -q pymupdf sentence-transformers transformers accelerate

import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import pymupdf
import re
import os

# --- Define necessary helper functions and constants within this cell for robustness ---

def read_pdf(pdf_path):
    document = pymupdf.open(pdf_path)
    pages_and_texts = []
    for page_number, page in enumerate(document):
        text = page.get_text("text")
        pages_and_texts.append({
            "page_number": page_number + 1,
            "text": text
        })
    return pages_and_texts

def clean_text(text):
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)
    text = text.replace("￾", "")
    return text.strip()

def split_into_sentences(text):
    sentences = re.split(
        r'(?<=[.!?])\s+',
        text
    )
    sentences = [
        sentence.strip()
        for sentence in sentences
        if sentence.strip()
    ]
    return sentences

NUM_SENTENCES_PER_CHUNK = 8

def create_sentence_chunks(
    sentences,
    chunk_size=NUM_SENTENCES_PER_CHUNK
):
    chunks = []
    for i in range(
        0,
        len(sentences),
        chunk_size
    ):
        chunk = sentences[
            i:i + chunk_size
        ]
        chunks.append(chunk)
    return chunks

MIN_TOKEN_LENGTH = 20 # From cells m3_VdgYS1Z9v and dYN597eT1dz_

# --- Re-generate pages_and_chunks, df, and embeddings from scratch within this cell ---
try:
    # Explicitly set pdf_path as 'Business.pdf' since it was confirmed to be uploaded.
    # This bypasses inconsistent global variable access issues.
    pdf_path = 'Business.pdf'
    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f"PDF file '{pdf_path}' not found. Please ensure the PDF is uploaded.")

    # Process PDF to get pages_and_texts
    pages_and_texts = read_pdf(pdf_path)

    # Clean and chunk text
    for item in pages_and_texts:
        item["text"] = clean_text(item["text"])
    for item in pages_and_texts:
        item["sentences"] = split_into_sentences(item["text"])
        item["sentence_count"] = len(item["sentences"])
    for item in pages_and_texts:
        item["sentence_chunks"] = create_sentence_chunks(item["sentences"])

    # Flatten into pages_and_chunks
    pages_and_chunks = []
    for item in pages_and_texts:
        for chunk in item["sentence_chunks"]:
            chunk_text = " ".join(chunk)
            pages_and_chunks.append({
                "page_number": item["page_number"],
                "sentence_chunk": chunk_text,
                "chunk_word_count": len(chunk_text.split()),
                "chunk_char_count": len(chunk_text)
            })

    # Create DataFrame and filter by token length
    df = pd.DataFrame(pages_and_chunks)
    df["chunk_token_count"] = df["chunk_char_count"] / 4 # Adding token count logic here
    df = df[df["chunk_token_count"] >= MIN_TOKEN_LENGTH].reset_index(drop=True)

    # Initialize embedding model
    embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

    # Generate embeddings
    texts = df["sentence_chunk"].tolist()
    embeddings = embedding_model.encode(
        texts,
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    df["embedding"] = list(embeddings)
    df.to_pickle("genai_book_embeddings.pkl")

    print("genai_book_embeddings.pkl recreated successfully and is now available.")

except Exception as e:
    print(f"An error occurred during embedding generation: {e}")
    raise

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/60 [00:00<?, ?it/s]

genai_book_embeddings.pkl recreated successfully and is now available.


In [ ]:
question = "What is mean?"
print("QUESTION:", question)

answer, sources = ask_book(question)

print("ANSWER:", answer)
print("\nSOURCES (Top 5 Retrieved Chunks):")
if sources:
    for j, source in enumerate(sources):
        print(f"  Source {j+1}: Page {source['page_number']}, Similarity: {round(source['score'], 3)}")
        print(f"    Text Snippet: {source['text'][:200]}...") # Print first 200 chars
else:
    print("  No sources retrieved.")

print("=" * 100)

In [ ]:
question = "What is variance?"
print("QUESTION:", question)

answer, sources = ask_book(question)

print("ANSWER:", answer)
print("\nSOURCES (Top 5 Retrieved Chunks):")
if sources:
    for j, source in enumerate(sources):
        print(f"  Source {j+1}: Page {source['page_number']}, Similarity: {round(source['score'], 3)}")
        print(f"    Text Snippet: {source['text'][:200]}...") # Print first 200 chars
else:
    print("  No sources retrieved.")

print("=" * 100)

In [ ]:
question = "What is mean?"
print("QUESTION:", question)

answer, sources = ask_book(question)

print("ANSWER:", answer)
print("\nSOURCES (Top 5 Retrieved Chunks):")
if sources:
    for j, source in enumerate(sources):
        print(f"  Source {j+1}: Page {source['page_number']}, Similarity: {round(source['score'], 3)}")
        print(f"    Text Snippet: {source['text'][:200]}...") # Print first 200 chars
else:
    print("  No sources retrieved.")

print("=" * 100)

QUESTION: What is mean?
ANSWER: data, we can calculate the mean.

SOURCES (Top 5 Retrieved Chunks):
  Source 1: Page 156, Similarity: 0.356
    Text Snippet: Interquartile Range 7KHGLIIHUHQFHEHWZHHQWKHYDOXHVRIWKH¿UVWDQGWKHWKLUGTXDUWLOHV WKLVGLIIHU- ence indicates the range of the middle half of the data set. Kurtosis The degree of peakedness o...
  Source 2: Page 148, Similarity: 0.327
    Text Snippet: The statistic TRMEAN is a “trimmed mean,” a mean calculated with the top 5 percent and bottom 5 percent of the data omitted. This helps to alleviate the distortion caused by the extreme values from wh...
  Source 3: Page 93, Similarity: 0.325
    Text Snippet: 78 Statistics for Management elements in a population is denoted by the capital italic letter N. Generally in statistics, we use italicized Roman letters to symbolize sample information and Greek lett...
  Source 4: Page 94, Similarity: 0.314
    Text Snippet: The formula looks like this: Sample Arithmetic Mean of Gro

In [ ]:
question = "What is mean?"
print("QUESTION:", question)

answer, sources = ask_book(question)

print("ANSWER:", answer)
print("\nSOURCES (Top 5 Retrieved Chunks):")
if sources:
    for j, source in enumerate(sources):
        print(f"  Source {j+1}: Page {source['page_number']}, Similarity: {round(source['score'], 3)}")
        print(f"    Text Snippet: {source['text'][:200]}...") # Print first 200 chars
else:
    print("  No sources retrieved.")

print("=" * 100)

QUESTION: What is mean?


NameError: name 'embeddings' is not defined

In [ ]:
question = "what is standard deviation formula?"
print("QUESTION:", question)

answer, sources = ask_book(question)

print("ANSWER:", answer)
print("\nSOURCES (Top 5 Retrieved Chunks):")
if sources:
    for j, source in enumerate(sources):
        print(f"  Source {j+1}: Page {source['page_number']}, Similarity: {round(source['score'], 3)}")
        print(f"    Text Snippet: {source['text'][:200]}...") # Print first 200 chars
else:
    print("  No sources retrieved.")

print("=" * 100)

QUESTION: what is standard deviation formula?
ANSWER: The standard deviation is the square root of the variance.

SOURCES (Top 5 Retrieved Chunks):
  Source 1: Page 135, Similarity: 0.705
    Text Snippet: 120 Statistics for Management FOHDURUHDVLO\LQWHUSUHWHG)RUWKLVUHDVRQZHKDYHWRPDNHDVLJQL¿FDQWFKDQJHLQWKHYDULDQFHWR compute a useful measure of deviation, one that does not give us a p...
  Source 2: Page 137, Similarity: 0.655
    Text Snippet: Notice that the distribution is reasonably symmetrical and that 93 percent is close to the theoretical 95 percent for an interval of plus and minus 2 standard devia- tions from the mean of a bell-shap...
  Source 3: Page 159, Similarity: 0.643
    Text Snippet: 144 Statistics for Management 3-12 x N x N ( ) 2 2 2 2 σ μ μ = Σ − = Σ − p. 119 This formula enables us to calculate the population variance, a measure of the average squared distance between the mean...
  Source 4: Page 135, Similarity: 0.64
    Text Snippet: When taki